![Token-Gated Language Models](https://raw.githubusercontent.com/token-gate-llms/TokenGate/main/assets/title.png)

[![GitHub](https://img.shields.io/badge/GitHub-TokenGate-181717?logo=github&logoColor=white)](https://github.com/token-gate-llms/TokenGate) [![arXiv](https://img.shields.io/badge/arXiv-2610.26754-B31B1B?logo=arxiv&logoColor=white)](https://arxiv.org/abs/2610.26754) [![Hugging Face paper](https://img.shields.io/badge/%F0%9F%A4%97-TokenGate%20paper-FFD21E)](https://huggingface.co/papers/2610.26754) [![X](https://img.shields.io/badge/X-Launch%20thread-000000?logo=x&logoColor=white)](https://x.com/lianapatel_/status/2112924355240099875) [![LinkedIn](https://img.shields.io/badge/LinkedIn-Post-0A66C2?logo=linkedin&logoColor=white)](https://lnkd.in/p/g3mj-QgZ)

## [0] Notebook overview

This notebook is a runnable introduction to TokenGate. We will:

1. **Set up** a needle-in-a-haystack task: hide a fact in a long document and ask for it.
2. **Run inference** with the standard model, then accelerate with TokenGate; compare answers and prefill time at several token budgets, and inspect the tokens selected by the gate.
3. **Train** a gate and a separate LoRA baseline on A100 GPUs, then compare them on held-out questions.

The default inference path runs on a free Colab T4; the training path needs A100 GPUs.



## What is TokenGate?

TokenGate is a small, drop-in component that extends existing trained models to enable end-to-end learnable token pruning over long-context inputs for fast and memory-efficient prefill.

### How token pruning works

A learned gate selects document tokens for later layers while keeping the question and instructions available.

![Animation: TokenGate selects document tokens after an early layer, so later layers process fewer tokens](https://raw.githubusercontent.com/token-gate-llms/TokenGate/61c7f5f769e1b9ddf498bb8b63c7174037cef37c/assets/tokengate_explainer.gif)

### Results

In [our paper](https://arxiv.org/abs/2610.26754), we report matched-accuracy results for Qwen3-1.7B on A100 GPUs against a separately LoRA-tuned full-prefill baseline. The chart on the left highlights NIAH and RepoBench; the Pareto plot on the right shows accuracy against prefill time (top) and memory (bottom) across five diverse tasks. Points toward the **upper left** mean better accuracy with less time or memory; red shows TokenGate, blue the tuned baseline, and gray other pruning and sparse attention methods.

<table><tr><td width="37%" valign="top"><img src="https://raw.githubusercontent.com/token-gate-llms/TokenGate/main/assets/results_figure.png" width="100%" alt="Published TokenGate speed and memory results for NIAH and RepoBench"></td><td width="63%" valign="top"><img src="https://raw.githubusercontent.com/token-gate-llms/TokenGate/0e7af59e4e8cbf7b6ad2db33f1875ce9eccf45f9/assets/tokengate_results_annotated.png" width="100%" alt="Accuracy versus time to first token and peak prefill memory across five tasks"></td></tr></table>


## [1] Setup and information

We will use **needle-in-a-haystack (NIAH)**. One sentence about an activity in a city is planted among unrelated essays; the model must answer a question about that sentence. The guided run uses one middle-depth example at **4k document tokens**. Its four gate budgets are **2,000, 500, 100, and 10**. After the guided run, an exploration cell lets you try different needle depths, document lengths, and budgets.

Choose the model and GPU runtime below, then install the repository API. A **token budget K** limits how many *document* tokens TokenGate passes to later layers; the question and instructions are always kept.

The inference cells follow the public API: prepare a prompt with `apply_chat_template`, restore the trained model with `load_model`, and call `generate`.

In [ ]:
#@title Runtime and model { display-mode: "form" }
MODEL = 'Qwen3-1.7B' #@param ['Qwen3-1.7B', 'Qwen3.5-2B']
GPU_RUNTIME = 'T4' #@param ['T4', 'A100']

BASE_MODELS = {'Qwen3-1.7B':'Qwen/Qwen3-1.7B', 'Qwen3.5-2B':'Qwen/Qwen3.5-2B'}
CHECKPOINTS = {'Qwen3-1.7B':('liana313/tokengate-niah-qwen3-1.7b','3ef4d84ebd161a9700acced6e92f7cd5c50225ad'),
               'Qwen3.5-2B':('liana313/tokengate-niah-qwen3.5-2b','b1fe3e72b9896efc821fbfa02a19fcded4de6d77')}
# Published full-prefill LoRA controls paired with the two released gates.
SFT_BASELINES = {
    'Qwen3-1.7B': ('liana313/tokengate-sftbase-niah-qwen3-1.7b',
                    '9a9703694aed601721414c3f8a6db7d9c83650c0', 'lora_step10200.pt'),
    'Qwen3.5-2B': ('liana313/tokengate-sftbase-niah-qwen3.5-2b',
                   '372728b32e8819964ec00279b1f45a64c803170d', 'lora_step600.pt'),
}
BASE_MODEL = BASE_MODELS[MODEL]
EXAMPLE_INDEX = 1  # middle-depth Hanoi example
DOC_TOKENS = 4096
BUDGETS = (2000, 500, 100, 10)
print('Guided run:', MODEL, '| middle-depth example | 4k document | K =', BUDGETS, '| GPU:', GPU_RUNTIME)


In [ ]:
#@title Install and import the API { display-mode: "form" }
import gc, importlib, logging, os, subprocess, sys, warnings
os.environ['HF_HUB_VERBOSITY'] = 'error'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', message='You are sending unauthenticated requests to the HF Hub.*')
from pathlib import Path
from IPython.utils.io import capture_output
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime first.'
assert GPU_RUNTIME in torch.cuda.get_device_name(0), f'Select a {GPU_RUNTIME} GPU in Colab: Runtime → Change runtime type.'
SOURCE = Path('/content/TokenGate') if Path('/content').is_dir() else Path.cwd()
if Path('/content').is_dir():
    if not SOURCE.exists():
        subprocess.run(['git','clone','-q','--branch','tokengate_demo',
                        'https://github.com/token-gate-llms/TokenGate',str(SOURCE)],check=True)
    else:
        subprocess.run(['git','-C',str(SOURCE),'fetch','origin','tokengate_demo'],check=True)
        subprocess.run(['git','-C',str(SOURCE),'switch','-C','tokengate_demo',
                        'origin/tokengate_demo'],check=True)
    local_head = subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip()
    remote_head = subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','origin/tokengate_demo'],text=True).strip()
    assert local_head == remote_head, 'The Colab checkout did not update to the latest demo branch.'
assert (SOURCE/'tokengate/demo.py').exists(), 'TokenGate demo helper is missing from this checkout.'
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(SOURCE),
                'transformers==5.5.0','matplotlib','ipywidgets>=8'],check=True)
sys.path.insert(0,str(SOURCE))
# A Colab runtime can keep the previous editable install in sys.modules.
for name in list(sys.modules):
    if name == 'tokengate' or name.startswith('tokengate.'):
        del sys.modules[name]
importlib.invalidate_caches()
from transformers import AutoTokenizer, AutoModelForCausalLM
if MODEL == 'Qwen3.5-2B':
    from transformers.models.qwen3_5.modeling_qwen3_5 import is_fast_path_available
    if not is_fast_path_available:
        logging.getLogger('transformers.models.qwen3_5.modeling_qwen3_5').setLevel(logging.ERROR)
        print('Qwen3.5: PyTorch DeltaNet fallback is active; baseline and gate timings include it.')
from huggingface_hub import snapshot_download
from tokengate import (apply_chat_template, split_state_span, load_model,
                       TokenGateConfig, TokenGateTrainer)
from tokengate.demo import (niah_examples, niah_prompt_text, niah_prompt,
                            plain_results, gate_results,
                            show_inference_results, show_kept_tokens, load_sft_baseline,
                            run_niah_exploration, release_niah_rows,
                            release_niah_config_kwargs, train_demo_sft_baseline)
print('GPU:', torch.cuda.get_device_name(0))

## [2] Inference

We will give the *same full document and question* to the base model and accelerated version with TokenGate. First, the notebook shows the middle-depth question and its expected answer. Then it runs both models, compares their answers with the planted fact, and measures prefill time.

### Without TokenGate: read the whole document

Behind the scenes, `{state}` marks the document TokenGate may prune. `apply_chat_template(tokenizer, prompt, state=document)` inserts that document and prepares token IDs. The question and instructions sit outside `{state}`, so the gate cannot remove them. `split_state_span` removes the gate's markers to give the ordinary model the same text.

The ordinary model reads the complete prompt. Its answer is our first check that the fact is retrievable; its **time to first token (TTFT)** is the timing reference for the TokenGate plots.


In [ ]:
# The same document and question go to both models.
from tokengate import apply_chat_template, split_state_span
sample = niah_examples(SOURCE, DOC_TOKENS)[EXAMPLE_INDEX]
with capture_output():
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)

# {state} marks the document that TokenGate may prune.
prompt_template = niah_prompt_text(sample)
gate_prompt = apply_chat_template(tok, prompt_template, state=sample['body'])
plain_prompt, state_start, state_end = split_state_span(gate_prompt)
samples, gate_prompts, plain_prompts = [sample], [gate_prompt], [plain_prompt]
print('Question:', sample['question'])
print('Expected answer:', sample['answer'])
print(f"Needle location: {sample['depth_actual']:.0%} into the document")
print(f'Document length: {state_end - state_start:,} tokens')


In [ ]:
#@title Run the fine-tuned full-prefill baseline { display-mode: "form" }
if GPU_RUNTIME == 'A100':
    assert torch.cuda.get_device_properties(0).total_memory >= 35*2**30, 'Use an A100-class GPU for single-pass prefill.'
dtype = torch.float16 if GPU_RUNTIME == 'T4' else torch.bfloat16
CHUNK_THRESHOLD = 4096  # Qwen3 on T4 uses cached chunks above 4k document tokens.
PLAIN_CHUNK_TOKENS = 1024 if GPU_RUNTIME == 'T4' and MODEL == 'Qwen3-1.7B' and DOC_TOKENS > CHUNK_THRESHOLD else None

# This demo helper loads the published LoRA-only control and folds its adapter
# into the base model. The TokenGate API starts in the next code cell.
baseline_id, baseline_revision, baseline_file = SFT_BASELINES[MODEL]
with capture_output():
    plain_model = load_sft_baseline(BASE_MODEL, baseline_id, baseline_file,
                                    revision=baseline_revision, dtype=dtype)
plain = plain_results(plain_model, tok, plain_prompts, samples,
                      chunk_tokens=PLAIN_CHUNK_TOKENS)
del plain_model
gc.collect()
torch.cuda.empty_cache()


### Accelerate with TokenGate

Now `load_model` restores the base model, trained gate, and paired LoRA weight updates as one object. We call `gate.generate` on the selected question, then measure this question at each budget.

`max_kept_prefill_tokens=K` limits *document* tokens passed to later layers. In the returned result, `answers` contains generated text, `n_state` counts document tokens presented to the gate, and `n_kept` counts all tokens kept for later layers, including the protected question and instructions. Smaller K can save work, but it may lose the fact. K=10 is included to test whether a very small budget loses the fact; compare every printed answer as well as the timing.


In [ ]:
# Public inference API: load a trained gate, then generate with a token budget.
from tokengate import load_model
repo_id, revision = CHECKPOINTS[MODEL]
with capture_output():
    checkpoint = snapshot_download(repo_id, revision=revision)
    gate = load_model(checkpoint, gate_attn='sdpa' if GPU_RUNTIME == 'T4' else 'flex', dtype=dtype)

# One call returns answers, prefill_s, n_kept, and kept_pos.
api_example = gate.generate([gate_prompt], tok, max_kept_prefill_tokens=BUDGETS[0],
                            max_new_tokens=64)
# Repeat at every K with warmup and correctness scoring for the result panel.
gated = gate_results(gate, tok, gate_prompts, samples, BUDGETS)
del gate
gc.collect()
torch.cuda.empty_cache()


### Compare answers and measurements

The table shows what each model actually answered, whether it found the planted fact, time to first token, and how much of the document and total prompt passed the gate. The token budget limits document tokens; the question and instructions stay available.

In [ ]:
#@title Answers and prefill comparison { display-mode: "form" }
baseline_name = ('chunked FT baseline (T4)' if PLAIN_CHUNK_TOKENS
                 else f'single-pass FT baseline ({GPU_RUNTIME})')
show_inference_results(plain, gated, baseline_name=baseline_name)


### Interpret the results

Read the ordinary answer and the answers at **K=2,000, 500, 100, and 10** first. The two plots show **TTFT**, where lower is faster, and **speedup**, where 2× means half the ordinary-model time. The takeaway identifies the smallest tested budget that retrieved the fact.

These live plots compare TokenGate with the **unmodified base model**. In our paper, we compare against a separately LoRA-tuned full-prefill baseline. On T4, the ordinary Qwen3 baseline uses cached chunks **above 4k** to fit in memory; our published measurements use single-pass prefill on A100. Qwen3.5 uses a single pass through 8k on T4. A100 uses a single-pass prefill for both the models.

### See what the gate selected

Choose one of the four tested budgets below to inspect the guided example. Highlighted spans are document tokens that survived pruning; the question and instructions were protected throughout. Compare the highlighted text with the expected answer printed above. This shows *what* the gate kept, although earlier layers may also have carried information from tokens it later dropped.


In [ ]:
#@title Which document tokens did the gate keep? { display-mode: "form" }
VIEW_BUDGET = '100' #@param ['2000', '500', '100', '10']
guided_result = next(r for r in gated if r['budget'] == int(VIEW_BUDGET))
show_kept_tokens(tok, gate_prompts[0], samples[0], guided_result)


### Try another example

Choose the **needle depth**, **document size**, and **token budgets** below. Run the settings cell in the current runtime to see the ordinary and TokenGate answers and the time and speedup plots. Saved notebook output alone does not restore its cached token positions. Then use the next cell to view the document tokens kept at any tested budget. Changing that dropdown only redraws saved token positions; it does not rerun inference.

The models load one at a time to fit a T4. At 16k on T4, Qwen3's ordinary baseline uses cached chunks above 4k, so its speedups differ from our paper's single-pass measurements but still give a relative sense of performance.


In [ ]:
#@title Demo settings: try another example { display-mode: "form" }
NEEDLE_DEPTH = 'Start' #@param ['Start', 'Middle', 'End']
DOCUMENT_SIZE = '8k' #@param ['2k', '4k', '8k', '16k']
TOKEN_BUDGETS = '2000, 500, 100, 10' #@param {type:"string"}

selected_index = {'Start': 0, 'Middle': 1, 'End': 2}[NEEDLE_DEPTH]
selected_doc_tokens = int(DOCUMENT_SIZE[:-1]) * 1024
selected_budgets = tuple(int(x.strip()) for x in TOKEN_BUDGETS.split(',') if x.strip())
assert selected_budgets and len(set(selected_budgets)) == len(selected_budgets) and all(k > 0 for k in selected_budgets)
if 'gate' in globals():
    del gate
    gc.collect(); torch.cuda.empty_cache()
import importlib
import tokengate.demo as demo_helpers
demo_helpers = importlib.reload(demo_helpers)
if getattr(demo_helpers, 'DEMO_CACHE_VERSION', 0) < 2:
    raise RuntimeError('Run Install and import the API above, then rerun Demo settings.')
repo_id, revision = CHECKPOINTS[MODEL]
exploration = demo_helpers.run_niah_exploration(
    SOURCE, base_model=BASE_MODEL, checkpoint_id=repo_id,
    checkpoint_revision=revision, baseline_checkpoint=SFT_BASELINES[MODEL],
    gpu_runtime=GPU_RUNTIME,
    doc_tokens=selected_doc_tokens, example_index=selected_index,
    budgets=selected_budgets)


In [ ]:
#@title View selected tokens { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display
import tokengate.demo as demo_helpers
if 'exploration' not in globals():
    raise RuntimeError('Run Demo settings in this runtime. Saved output does not restore its results.')
if exploration.get('cache_version', 0) < 2 or 'tokenizer' not in exploration or any('kept_pos' not in row for row in exploration['gated']):
    raise RuntimeError('These results predate cached token selections. Rerun Install and import the API, then Demo settings.')

budget_options = tuple(dict.fromkeys(row['budget'] for row in exploration['gated']))
selected_token_budget = widgets.Dropdown(
    options=[(f'K={k:,}', k) for k in budget_options],
    value=100 if 100 in budget_options else sorted(budget_options)[len(budget_options) // 2],
    description='View selected tokens:',
    style={'description_width': 'initial'})

def render_selected_tokens(token_budget):
    result = next(row for row in exploration['gated'] if row['budget'] == token_budget)
    demo_helpers.show_kept_tokens(exploration['tokenizer'], exploration['gate_prompt'],
                     exploration['sample'], result)

token_view = widgets.interactive(render_selected_tokens,
                                 token_budget=selected_token_budget)
display(token_view)


## [3] Optional: train and compare your own models

The results above use released weights. This optional A100 section trains a new gate and paired LoRA updates on 12,000 NIAH rows, using 50 validation rows and holding 150 rows out for testing. It also trains a separate LoRA-only model so the comparison uses the same kind of full-prefill baseline as our paper. Four A100s match our published batch layout; fewer GPUs take longer, and a full run can take hours.

To run it, switch to an A100 runtime and enable `RUN_TRAINING`. The code shows the public training pattern: put the recipe settings in `TokenGateConfig`, pass `article` and `answer` rows to `TokenGateTrainer`, and call `trainer.train()`. The final cell loads both trained models and compares their answers and prefill times on the selected held-out question at 16k. That quick check is smaller than the full evaluation in our paper.


In [ ]:
# Train TokenGate and a separate full-prefill LoRA control (A100 only).
RUN_TRAINING = False #@param {type:"boolean"}
if RUN_TRAINING:
    from tokengate import TokenGateConfig, TokenGateTrainer
    assert GPU_RUNTIME == 'A100', 'Choose A100 above and switch to an A100 runtime.'
    n_gpus = torch.cuda.device_count()
    assert n_gpus in (1, 2, 4) and all('A100' in torch.cuda.get_device_name(i) for i in range(n_gpus)), 'Use 1, 2, or 4 visible A100s.'
    gc.collect()
    torch.cuda.empty_cache()
    work = Path('/content/tokengate-training') if Path('/content').is_dir() else Path.cwd()/'tokengate-training'
    work = work/MODEL
    data_root = work/'data'

    # Replace these NIAH rows with your own {"article": text, "answer": text} rows.
    train_rows, eval_rows = release_niah_rows(SOURCE, work)
    recipe = release_niah_config_kwargs(SOURCE, BASE_MODEL, work/'gate', data_root, n_gpus)

    # These three public API calls are the reusable training pattern.
    config = TokenGateConfig(**recipe)
    trainer = TokenGateTrainer(config, train_dataset=train_rows,
                               eval_dataset=eval_rows, task='niah_kamradt')
    gate_checkpoint = Path(config.output_dir)/'gate_final.pt'
    if not gate_checkpoint.exists():
        if MODEL == 'Qwen3-1.7B':
            from kernels.fa2bias.keybias import available as fa2_available
            if not fa2_available():
                subprocess.run([sys.executable, str(SOURCE/'kernels/fa2bias/build.py')], check=True)
        trainer.train()
    del trainer, train_rows, eval_rows
    gc.collect()

    # A separate LoRA-only control uses the same generated rows and split.
    baseline_dir = train_demo_sft_baseline(
        BASE_MODEL, data_root, config.eval_split, work/'lora_baseline', n_gpus)
    print('Gate:', gate_checkpoint, '| LoRA baseline:', baseline_dir)
else:
    print('Optional full training skipped; inference above used the released checkpoints.')


In [ ]:
#@title Compare your trained gate with its LoRA baseline (A100 only) { display-mode: "form" }
if RUN_TRAINING:
    from peft import PeftModel
    benchmark_rows = [niah_examples(SOURCE, 16384)[EXAMPLE_INDEX]]
    pairs = [niah_prompt(tok, row) for row in benchmark_rows]
    trained_gate_prompts, baseline_prompts = map(list, zip(*pairs))
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16, attn_implementation='sdpa')
    tuned_base = PeftModel.from_pretrained(base, str(baseline_dir)).merge_and_unload().cuda().eval()
    tuned_plain = plain_results(tuned_base, tok, baseline_prompts, benchmark_rows)
    print('LoRA baseline:', sum(r['correct'] for r in tuned_plain), '/', len(tuned_plain), 'correct')
    del tuned_base, base
    gc.collect(); torch.cuda.empty_cache()
    own_gate = load_model(str(gate_checkpoint), gate_attn='fa2' if MODEL == 'Qwen3-1.7B' else 'flex', dtype=torch.bfloat16)
    own = gate_results(own_gate, tok, trained_gate_prompts, benchmark_rows, BUDGETS)
    for k in BUDGETS:
        rows = [r for r in own if r['budget'] == k]
        print(f'K={k}: {sum(r["correct"] for r in rows)}/{len(rows)} correct')
    show_inference_results(tuned_plain, own, baseline_name='newly trained LoRA baseline (A100)')
else:
    print('Run the optional training cell first.')
